In [1]:
import cv2
import numpy as np
import easyocr
import pandas as pd
from rapidfuzz import fuzz
import re
import unicodedata
import sys
import os

In [2]:
IMG_PATH     = "dataset/images/oradexon.jpg"
CSV_PATH     = "dataset/medecaments.csv"
CSV_SEP      = ";"
CSV_ENCODING = "utf-8"   
DEBUG        = True
SEUIL_REJET  = 50        

# Colonnes CSV
COL_CODE        = "CODE"
COL_NOM_FR      = "NOM"
COL_NOM_AR      = "الاسم"
COL_MOLECULE_FR = "DCI1"
COL_MOLECULE_AR = "معلومات الدواء"
COL_DOSAGE      = "DOSAGE1"
COL_UNITE_FR    = "UNITE_DOSAGE1"
COL_UNITE_AR    = "وحدة الجرعة"
COL_FORME_FR    = "FORME"
COL_FORME_AR    = "صيغة"
COL_COND_FR     = "PRESENTATION"
COL_COND_AR     = "شكل الدواء"

# Toutes les colonnes attendues
TOUTES_COLONNES = {
    COL_CODE:        "Code produit",
    COL_NOM_FR:      "Nom (FR)",
    COL_NOM_AR:      "Nom (AR)",
    COL_MOLECULE_FR: "DCI / Molécule (FR)",
    COL_MOLECULE_AR: "DCI / Molécule (AR)",
    COL_DOSAGE:      "Dosage",
    COL_UNITE_FR:    "Unité (FR)",
    COL_UNITE_AR:    "Unité (AR)",
    COL_FORME_FR:    "Forme (FR)",
    COL_FORME_AR:    "Forme (AR)",
    COL_COND_FR:     "Présentation (FR)",
    COL_COND_AR:     "Présentation (AR)",
}

W_NAME   = 0.70
W_DOSAGE = 0.20
W_FORME  = 0.10

FORMS_KW = [
    "COMPRIME", "COMPRIMES", "COMPRIME PELLICULE", "COMPRIME ENROBE",
    "COMPRIME SECABLE", "COMPRIME EFFERVESCENT",
    "GELULE", "GELULES",
    "SIROP", "FLACON", "POMMADE", "SOLUTION", "SOLUTION BUVABLE",
    "CAPSULE", "SACHET", "INJECTABLE", "SUSPENSION",
    "PATCH", "CREME", "GEL", "SPRAY", "POUDRE",
    "POMMADE OPHTALMIQUE", "COLLYRE", "SUPPOSITOIRE",
    "LYOPHILISAT", "EMULSION", "LOTION", "MOUSSE",
    "GRANULES", "GRANULE", "AMPOULE",
]
FORMS_KW.sort(key=len, reverse=True)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# UTILITAIRES
# ─────────────────────────────────────────────────────────────────────────────
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_latin(text: str) -> str:
    text = str(text).translate(ARABIC_DIGITS)
    text = unicodedata.normalize("NFD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.upper()
    text = re.sub(r"[^A-Z0-9 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_arabic(text: str) -> str:
    text = str(text).translate(ARABIC_DIGITS)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)   # supprime tashkeel
    text = re.sub(r"[^\u0600-\u06FF\u0750-\u077F 0-9]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def safe(row, col: str, fallback: str = "") -> str:
    val = row.get(col, fallback)
    return "" if str(val).strip() in ("nan", "NaN", "") else str(val).strip()

def log(msg: str):
    if DEBUG:
        print(msg)

def ligne(car: str = "─", n: int = 55) -> str:
    return car * n


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. CHARGEMENT IMAGE
# ─────────────────────────────────────────────────────────────────────────────
print(ligne("="))
print("  PIPELINE OCR ROBUSTE – MÉDICAMENTS  (v2)")
print(ligne("="))

if not os.path.exists(IMG_PATH):
    print(f"\n❌ Image introuvable : {IMG_PATH}")
    sys.exit(1)

img = cv2.imread(IMG_PATH)
h_orig, w_orig = img.shape[:2]
log(f"\n✔ Image chargée  ({w_orig}×{h_orig} px)")


  PIPELINE OCR ROBUSTE – MÉDICAMENTS  (v2)

✔ Image chargée  (169×298 px)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. PREPROCESSING ADAPTATIF
# ─────────────────────────────────────────────────────────────────────────────
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

scale = max(1.0, 1200 / h_orig)
if scale > 1.0:
    gray = cv2.resize(gray, None, fx=scale, fy=scale,
                      interpolation=cv2.INTER_CUBIC)
    log(f"✔ Redimensionnement ×{scale:.2f}")

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
gray  = clahe.apply(gray)
gray  = cv2.GaussianBlur(gray, (3, 3), 0)

if np.mean(gray) < 100:
    gray = cv2.bitwise_not(gray)
    log("✔ Fond sombre → inversion")

_, otsu  = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 15, 4)
gray_bin = cv2.bitwise_or(otsu, adaptive)

os.makedirs("dataset/images", exist_ok=True)
cv2.imwrite("dataset/images/processed_gray.jpg", gray)
cv2.imwrite("dataset/images/processed_bin.jpg",  gray_bin)
log("✔ Preprocessing terminé")

✔ Redimensionnement ×4.03
✔ Preprocessing terminé


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. OCR BILINGUE
# ─────────────────────────────────────────────────────────────────────────────
log("\n✔ Lancement OCR…")

reader_fr = easyocr.Reader(["fr", "en"], gpu=False)
reader_ar = easyocr.Reader(["ar"],       gpu=False)

def run_ocr(reader, paths, threshold=0.35):
    out = []
    for p in paths:
        for (_, text, conf) in reader.readtext(p):
            if conf >= threshold:
                out.append((text, conf))
    return out

imgs   = ["dataset/images/processed_gray.jpg",
          "dataset/images/processed_bin.jpg"]
res_fr = run_ocr(reader_fr, imgs, threshold=0.35)
res_ar = run_ocr(reader_ar, imgs, threshold=0.40)

raw_latin  = " ".join(t for t, _ in res_fr)
ocr_latin  = normalize_latin(raw_latin)
raw_arabic = " ".join(t for t, _ in res_ar)
ocr_arabic = normalize_arabic(raw_arabic)

log(f"\n{ligne()}")
log("OCR LATIN  : " + (ocr_latin[:200]  or "(rien détecté)"))
log("OCR ARABE  : " + (ocr_arabic[:100] if ocr_arabic else "(rien détecté)"))
log(ligne())

ocr_words   = ocr_latin.split()
ocr_numbers = [w for w in ocr_words if any(c.isdigit() for c in w)]

ocr_forms_detected = [kw for kw in FORMS_KW if kw in ocr_latin]

log(f"Nombres détectés  : {ocr_numbers or ['aucun']}")
log(f"Formes détectées  : {ocr_forms_detected or ['aucune (sera compensé par score direct)']}\n")

Using CPU. Note: This module is much faster with a GPU.



✔ Lancement OCR…


Using CPU. Note: This module is much faster with a GPU.
c:\Users\khaou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\khaou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\khaou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\khaou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerato


───────────────────────────────────────────────────────
OCR LATIN  : ORADEXONOS M 20 COMPIMES VOLE ORALO ORADEXONOS IG 20 COMNPRIMES VOLE ORALC 0 3 1 8
OCR ARABE  : (rien détecté)
───────────────────────────────────────────────────────
Nombres détectés  : ['20', '20', '0', '3', '1', '8']
Formes détectées  : ['aucune (sera compensé par score direct)']



In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. CHARGEMENT DATASET + DIAGNOSTIC COLONNES
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CSV_PATH):
    print(f"\n❌ CSV introuvable : {CSV_PATH}")
    sys.exit(1)

df = None
for enc in [CSV_ENCODING, "utf-8-sig", "windows-1256", "latin-1"]:
    try:
        df = pd.read_csv(CSV_PATH, sep=CSV_SEP, encoding=enc, dtype=str)
        df = df.fillna("")
        df.columns = df.columns.str.strip()
        for col in df.columns:
            df[col] = df[col].str.strip()
        log(f"✔ CSV chargé avec encodage : {enc}  ({len(df)} médicaments)")
        break
    except Exception:
        continue

if df is None:
    print("❌ Impossible de lire le CSV — vérifie le séparateur et l'encodage")
    sys.exit(1)

print(f"\n{ligne('─')}")
print("  DIAGNOSTIC COLONNES CSV")
print(ligne("─"))

cols_csv = list(df.columns)
for col, label in TOUTES_COLONNES.items():
    if col not in cols_csv:
        statut  = "❌ COLONNE ABSENTE"
        conseil = "→ Vérifier le nom exact dans le CSV"
    else:
        nb_vides   = df[col].eq("").sum()
        nb_total   = len(df)
        pct_rempli = (nb_total - nb_vides) / nb_total * 100
        if pct_rempli == 0:
            statut  = "⚠️  VIDE (0% rempli)"
            conseil = "→ Cette colonne n'a aucune donnée : affichera toujours —"
        elif pct_rempli < 50:
            statut  = f"⚠️  {pct_rempli:.0f}% rempli"
            conseil = "→ Données partielles"
        else:
            statut  = f"✔  {pct_rempli:.0f}% rempli"
            conseil = ""
    print(f"  {label:<26} {statut}  {conseil}")

print(ligne("─"))
print("\n  Exemple ligne 1 du CSV :")
row0 = df.iloc[0]
for col, label in TOUTES_COLONNES.items():
    val = row0.get(col, "COLONNE ABSENTE")
    val_display = (
        val
        if any('\u0600' <= c <= '\u06FF' for c in str(val))
        else (val or "—")
    )
    print(f"    {label:<26} {val_display}")
print(ligne("─") + "\n")

✔ CSV chargé avec encodage : utf-8  (100 médicaments)

───────────────────────────────────────────────────────
  DIAGNOSTIC COLONNES CSV
───────────────────────────────────────────────────────
  Code produit               ✔  100% rempli  
  Nom (FR)                   ✔  100% rempli  
  Nom (AR)                   ✔  100% rempli  
  DCI / Molécule (FR)        ✔  100% rempli  
  DCI / Molécule (AR)        ✔  100% rempli  
  Dosage                     ✔  97% rempli  
  Unité (FR)                 ✔  98% rempli  
  Unité (AR)                 ✔  99% rempli  
  Forme (FR)                 ✔  100% rempli  
  Forme (AR)                 ✔  100% rempli  
  Présentation (FR)          ✔  100% rempli  
  Présentation (AR)          ✔  100% rempli  
───────────────────────────────────────────────────────

  Exemple ligne 1 du CSV :
    Code produit               6118000170211
    Nom (FR)                   GYNO CANESTEN
    Nom (AR)                   جينو كانستين
    DCI / Molécule (FR)        CLOTRIMAZ

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. FONCTIONS DE SCORING
# ─────────────────────────────────────────────────────────────────────────────
def score_nom_latin(ocr: str, nom: str) -> float:
    if not nom:
        return 0.0
    n = normalize_latin(nom)
    if not n:
        return 0.0
    return float(max(
        fuzz.token_set_ratio(ocr, n),
        fuzz.partial_ratio(ocr, n),
        fuzz.token_sort_ratio(ocr, n),
    ))

def score_nom_arabe(ocr_ar: str, nom_ar: str) -> float:
    if not nom_ar or not ocr_ar:
        return 0.0
    n = normalize_arabic(nom_ar)
    if not n:
        return 0.0
    return float(max(
        fuzz.token_set_ratio(ocr_ar, n),
        fuzz.partial_ratio(ocr_ar, n),
    ))

def score_dosage(ocr_nums: list, dosage_str: str) -> float:
    if not dosage_str or not ocr_nums:
        return 0.0
    d = normalize_latin(dosage_str)
    best = 0.0
    for n in ocr_nums:
        if len(n) == 1 and n.isdigit():
            s = float(fuzz.ratio(n, d))
        else:
            s = float(fuzz.partial_ratio(n, d))
        if s > best:
            best = s
    return best

def score_forme(ocr_full: str, ocr_forms_kw: list, forme_str: str) -> float:
    """
    Double stratégie :
      1. Comparaison directe entre le texte OCR complet et la forme du CSV
         → fonctionne même si aucun mot-clé n'a été extrait
      2. Comparaison via les mots-clés détectés dans l'OCR
         → donne un bonus si un mot-clé exact est présent
    """
    if not forme_str:
        return 0.0
    f = normalize_latin(forme_str)
    if not f:
        return 0.0

    score_direct = float(fuzz.partial_ratio(ocr_full, f))

    score_kw = 0.0
    if ocr_forms_kw:
        score_kw = float(max(fuzz.partial_ratio(kw, f) for kw in ocr_forms_kw))

    final = max(score_direct, score_kw)
    return final

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. MATCHING ENGINE
# ─────────────────────────────────────────────────────────────────────────────
log("✔ Matching en cours…")
results = []

for _, row in df.iterrows():
    nom_fr   = safe(row, COL_NOM_FR)
    nom_ar   = safe(row, COL_NOM_AR)
    mol_fr   = safe(row, COL_MOLECULE_FR)
    mol_ar   = safe(row, COL_MOLECULE_AR)
    dosage   = safe(row, COL_DOSAGE)
    unite_fr = safe(row, COL_UNITE_FR)
    unite_ar = safe(row, COL_UNITE_AR)
    forme_fr = safe(row, COL_FORME_FR)
    forme_ar = safe(row, COL_FORME_AR)
    cond_fr  = safe(row, COL_COND_FR)
    cond_ar  = safe(row, COL_COND_AR)
    code     = safe(row, COL_CODE)

    sn = max(
        score_nom_latin(ocr_latin,  nom_fr),
        score_nom_latin(ocr_latin,  mol_fr),
        score_nom_arabe(ocr_arabic, nom_ar),
        score_nom_arabe(ocr_arabic, mol_ar),
    )
    sd = score_dosage(ocr_numbers, dosage)

    sf = score_forme(ocr_latin, ocr_forms_detected, forme_fr)

    final = W_NAME * sn + W_DOSAGE * sd + W_FORME * sf

    results.append({
        "code":     code,
        "nom_fr":   nom_fr,   "nom_ar":   nom_ar,
        "mol_fr":   mol_fr,   "mol_ar":   mol_ar,
        "dosage":   dosage,
        "unite_fr": unite_fr, "unite_ar": unite_ar,
        "forme_fr": forme_fr, "forme_ar": forme_ar,
        "cond_fr":  cond_fr,  "cond_ar":  cond_ar,
        "sn": sn, "sd": sd, "sf": sf,
        "final": final,
    })

results.sort(key=lambda x: x["final"], reverse=True)
best = results[0]

✔ Matching en cours…


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. AFFICHAGE TOP 5
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{ligne('=')}")
print("  TOP 5 CORRESPONDANCES")
print(ligne("="))
print(f"  {'#':<3} {'NOM':<28} {'NOM':>6} {'DOS':>6} {'FOR':>6} {'FINAL':>7}")
print(f"  {ligne(n=53)}")
for i, r in enumerate(results[:5], 1):
    m = "►" if i == 1 else " "
    print(
        f"  {m}#{i}  {r['nom_fr']:<28}"
        f"  {r['sn']:5.1f}"
        f"  {r['sd']:5.1f}"
        f"  {r['sf']:5.1f}"
        f"  {r['final']:6.1f}"
    )


  TOP 5 CORRESPONDANCES
  #   NOM                             NOM    DOS    FOR   FINAL
  ─────────────────────────────────────────────────────
  ►#1  ORADEXON                      100.0   66.7   68.8    90.2
   #2  TOBRADEX                       85.7   50.0   42.1    74.2
   #3  IMODIUM                        66.7  100.0   33.3    70.0
   #4  NOLVADEX                       71.4   66.7   61.1    69.4
   #5  TRUSOPT                        62.5  100.0   50.0    68.8


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. SEUIL DE REJET + FICHE COMPLÈTE
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{ligne('═')}")

if best["final"] < SEUIL_REJET:
    print("  ❌  MÉDICAMENT NON RECONNU")
    print(ligne("═"))
    print()
    print(f"  Score maximum obtenu  : {best['final']:.2f} / 100")
    print(f"  Seuil minimum requis  : {SEUIL_REJET} / 100")
    print()
    print("  Raisons possibles :")
    print("   • Ce médicament n'est pas dans la base de données")
    print("   • La photo est trop floue ou mal cadrée")
    print("   • Le texte est trop peu visible pour l'OCR")
    print()
    print("  Solutions :")
    print("   1. Reprendre la photo (lumière directe, angle droit à la boîte)")
    print("   2. Ajouter ce médicament dans le fichier CSV")
    print("   3. Baisser SEUIL_REJET si la base de données est incomplète")
    print()
    if best["final"] >= 35:
        print("  Suggestion (non confirmée) :")
        print(f"   Médicament le plus proche : {best['nom_fr']}")
        print(f"   Score                     : {best['final']:.1f} / 100")
        print("   → Vérifier manuellement avant toute utilisation")
    print(f"\n{ligne('═')}\n")

else:
    print("  FICHE DU MÉDICAMENT IDENTIFIÉ")
    print(ligne("═"))

    if   best["final"] >= 75: conf_txt = "HAUTE    ✅"
    elif best["final"] >= 60: conf_txt = "BONNE    ✅"
    else:                      conf_txt = "MOYENNE  ⚠️  (vérifier si nécessaire)"

    def affiche_champ(label_fr, label_ar, val_fr, val_ar_brut):
        v_ar = val_ar_brut
        print(f"\n  {ligne(n=51)}")
        print(f"  {label_fr:<26} {val_fr or '—'}")
        if v_ar != "—":
            print(f"  {label_ar:<26} {v_ar}")
        else:
            print(f"  {label_ar:<26} — (non renseigné dans la base)")

    # Nom
    print(f"\n  {'NOM (FR)':<26} {best['nom_fr'] or '—'}")
    v_nom_ar = best['nom_ar']
    if v_nom_ar != "—":
        print(f"  {'NOM (AR)':<26} {v_nom_ar}")
    else:
        print(f"  {'NOM (AR)':<26} — (non renseigné dans la base)")

    # DCI
    affiche_champ(
        "DCI / PRINCIPE ACTIF (FR)", "DCI / PRINCIPE ACTIF (AR)",
        best["mol_fr"], best["mol_ar"]
    )

    # Dosage
    uf     = best["unite_fr"] or ""
    ua_raw = best["unite_ar"]
    ua     = ua_raw if ua_raw else ""
    dos_fr = f"{best['dosage']} {uf}".strip() if best["dosage"] else "—"
    dos_ar = f"{best['dosage']} {ua}".strip() if (best["dosage"] and ua) else (best["dosage"] or "—")
    print(f"\n  {ligne(n=51)}")
    print(f"  {'DOSAGE (FR)':<26} {dos_fr}")
    print(f"  {'DOSAGE (AR)':<26} {dos_ar if ua else dos_fr + '  (unité AR non renseignée)'}")

    # Forme
    affiche_champ("FORME (FR)", "FORME (AR)", best["forme_fr"], best["forme_ar"])

    # Présentation
    affiche_champ("PRÉSENTATION (FR)", "PRÉSENTATION (AR)", best["cond_fr"], best["cond_ar"])

    # Code + scores
    print(f"\n  {ligne(n=51)}")
    print(f"  {'CODE PRODUIT':<26} {best['code'] or '—'}")
    print(f"  {ligne(n=51)}")
    print(f"  {'SCORE NOM':<26} {best['sn']:.2f} / 100")
    print(f"  {'SCORE DOSAGE':<26} {best['sd']:.2f} / 100")
    print(f"  {'SCORE FORME':<26} {best['sf']:.2f} / 100")
    print(f"  {ligne(n=51)}")
    print(f"  {'SCORE FINAL':<26} {best['final']:.2f} / 100")
    print(f"  {'CONFIANCE':<26} {conf_txt}")
    print(f"\n{ligne('═')}\n")


═══════════════════════════════════════════════════════
  FICHE DU MÉDICAMENT IDENTIFIÉ
═══════════════════════════════════════════════════════

  NOM (FR)                   ORADEXON
  NOM (AR)                   أوراديكسون

  ───────────────────────────────────────────────────
  DCI / PRINCIPE ACTIF (FR)  DEXAMETHASONE
  DCI / PRINCIPE ACTIF (AR)  ديكساميثازون

  ───────────────────────────────────────────────────
  DOSAGE (FR)                0.5 MG
  DOSAGE (AR)                0.5 ملغ

  ───────────────────────────────────────────────────
  FORME (FR)                 COMPRIME SECABLE
  FORME (AR)                 أقراص قابلة للكسر

  ───────────────────────────────────────────────────
  PRÉSENTATION (FR)          1 BOITE 20 COMPRIME
  PRÉSENTATION (AR)          علبة واحدة تحتوي على 20 قرصًا

  ───────────────────────────────────────────────────
  CODE PRODUIT               6118000080787
  ───────────────────────────────────────────────────
  SCORE NOM                  100.00 / 100
  S